In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Gün 2'de eğittiğimiz modeli geri yükle
model = keras.models.load_model("../model/hareket_modeli.keras")
print("Model yüklendi.")
model.summary()

# Kuantizasyon için veriye de ihtiyacımız var (6 kanal) — tekrar yükleyelim
VERI_YOLU = "../data/UCI HAR Dataset"

def sinyal_yukle(dosya_adi, bolum):
    yol = f"{VERI_YOLU}/{bolum}/Inertial Signals/{dosya_adi}"
    return pd.read_csv(yol, sep=r"\s+", header=None).values

kanallar_tr = [
    sinyal_yukle("body_acc_x_train.txt", "train"),
    sinyal_yukle("body_acc_y_train.txt", "train"),
    sinyal_yukle("body_acc_z_train.txt", "train"),
    sinyal_yukle("body_gyro_x_train.txt", "train"),
    sinyal_yukle("body_gyro_y_train.txt", "train"),
    sinyal_yukle("body_gyro_z_train.txt", "train"),
]
kanallar_te = [
    sinyal_yukle("body_acc_x_test.txt", "test"),
    sinyal_yukle("body_acc_y_test.txt", "test"),
    sinyal_yukle("body_acc_z_test.txt", "test"),
    sinyal_yukle("body_gyro_x_test.txt", "test"),
    sinyal_yukle("body_gyro_y_test.txt", "test"),
    sinyal_yukle("body_gyro_z_test.txt", "test"),
]

X_train = np.stack(kanallar_tr, axis=-1).astype("float32")
X_test  = np.stack(kanallar_te, axis=-1).astype("float32")
y_test  = pd.read_csv(f"{VERI_YOLU}/test/y_test.txt", header=None).values.ravel() - 1

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

Model yüklendi.


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 126, 16)        │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 63, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 61, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 30, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,380 (36.64 KB)

 Trainable params: 3,126 (12.21 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,254 (24.43 KB)

X_train: (7352, 128, 6) | X_test: (2947, 128, 6)


In [4]:
# --- int8 KUANTİZASYON ---
# Kuantizasyon aracına "temsili veri" veriyoruz: modelin göreceği tipik
# girdilerden birkaç yüz örnek. Araç bunlara bakarak sayıları nasıl
# int8'e sıkıştıracağını (hangi aralıkta çalıştığını) öğrenir.

def temsili_veri():
    for i in range(200):                      # 200 örnek yeterli
        ornek = X_train[i:i+1]                 # tek örnek, (1, 128, 6)
        yield [ornek.astype("float32")]

# Dönüştürücüyü hazırla
donusturucu = tf.lite.TFLiteConverter.from_keras_model(model)
donusturucu.optimizations = [tf.lite.Optimize.DEFAULT]        # kuantizasyonu aç
donusturucu.representative_dataset = temsili_veri             # temsili veriyi ver

# Tam int8 kuantizasyon: girdi/çıktı dahil her şey int8
donusturucu.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
donusturucu.inference_input_type = tf.int8
donusturucu.inference_output_type = tf.int8

# Dönüştür
tflite_model = donusturucu.convert()

# Kaydet
with open("../model/hareket_modeli_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("Kuantize model kaydedildi.")
print(f"int8 model boyutu: {len(tflite_model)} bayt  ({len(tflite_model)/1024:.1f} KB)")

INFO:tensorflow:Assets written to: C:\Users\MERFAR~1\AppData\Local\Temp\tmpy66c4390\assets


INFO:tensorflow:Assets written to: C:\Users\MERFAR~1\AppData\Local\Temp\tmpy66c4390\assets


Saved artifact at 'C:\Users\MERFAR~1\AppData\Local\Temp\tmpy66c4390'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1906214979536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214981456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214979344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214980880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214983376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214984144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214983568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214984720: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\staj_projesi4\tinyml-hareket-tanima\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Kuantize model kaydedildi.
int8 model boyutu: 12448 bayt  (12.2 KB)


In [5]:
# --- İYİLEŞTİRİLMİŞ int8 KUANTİZASYON ---
# Fark: temsili veriyi RASTGELE ve DAHA ÇOK örnekten seçiyoruz.
# Böylece araç modelin tüm hareket çeşitliliğini görüp daha isabetli kalibre eder.

rng = np.random.default_rng(42)                      # tekrarlanabilir rastgelelik
secili = rng.choice(len(X_train), size=500, replace=False)   # 500 rastgele örnek

def temsili_veri_v2():
    for i in secili:
        ornek = X_train[i:i+1].astype("float32")
        yield [ornek]

donusturucu = tf.lite.TFLiteConverter.from_keras_model(model)
donusturucu.optimizations = [tf.lite.Optimize.DEFAULT]
donusturucu.representative_dataset = temsili_veri_v2
donusturucu.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
donusturucu.inference_input_type = tf.int8
donusturucu.inference_output_type = tf.int8

tflite_model_v2 = donusturucu.convert()

with open("../model/hareket_modeli_int8.tflite", "wb") as f:
    f.write(tflite_model_v2)

print(f"Yeni int8 model boyutu: {len(tflite_model_v2)/1024:.1f} KB")
print("Şimdi doğruluğu tekrar ölçelim...")

INFO:tensorflow:Assets written to: C:\Users\MERFAR~1\AppData\Local\Temp\tmp4qfzl70r\assets


INFO:tensorflow:Assets written to: C:\Users\MERFAR~1\AppData\Local\Temp\tmp4qfzl70r\assets


Saved artifact at 'C:\Users\MERFAR~1\AppData\Local\Temp\tmp4qfzl70r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1906214979536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214981456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214979344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214980880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214983376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214984144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214983568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1906214984720: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\staj_projesi4\tinyml-hareket-tanima\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Yeni int8 model boyutu: 12.2 KB
Şimdi doğruluğu tekrar ölçelim...


In [6]:
# --- KUANTİZE MODELİN DOĞRULUĞUNU ÖLÇ ---
# .tflite modelini "interpreter" ile çalıştırıp test setinde deniyoruz.

interpreter = tf.lite.Interpreter(model_path="../model/hareket_modeli_int8.tflite")
interpreter.allocate_tensors()

girdi = interpreter.get_input_details()[0]
cikti = interpreter.get_output_details()[0]

print("Girdi tipi:", girdi["dtype"])          # int8 olmalı
print("Girdi ölçek/sıfır:", girdi["quantization"])

# Girdiyi int8'e çevirmek için kuantizasyon parametreleri
giris_olcek, giris_sifir = girdi["quantization"]

dogru = 0
for i in range(len(X_test)):
    # float örneği int8'e kuantize et
    x = X_test[i:i+1] / giris_olcek + giris_sifir
    x = np.round(x).astype(np.int8)

    interpreter.set_tensor(girdi["index"], x)
    interpreter.invoke()
    sonuc = interpreter.get_tensor(cikti["index"])

    tahmin = np.argmax(sonuc[0])
    if tahmin == y_test[i]:
        dogru += 1

int8_dogruluk = dogru / len(X_test)
print(f"\nKuantize (int8) model doğruluğu: {int8_dogruluk:.4f}  ({int8_dogruluk*100:.1f}%)")
print("Karşılaştırma → float model: ~86.1%")

c:\staj_projesi4\tinyml-hareket-tanima\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Girdi tipi: <class 'numpy.int8'>
Girdi ölçek/sıfır: (0.031152410432696342, -14)

Kuantize (int8) model doğruluğu: 0.8140  (81.4%)
Karşılaştırma → float model: ~86.1%


In [7]:
# --- .tflite MODELİNİ C DİZİSİNE ÇEVİR (model_data.h) ---
# Çip Python/tflite dosyası okuyamaz; modeli C kodundaki bir bayt dizisine
# gömeceğiz. Bu dosyayı sonra ESP32 firmware'ine dahil edeceğiz.

# Kaydettiğimiz int8 modeli oku
with open("../model/hareket_modeli_int8.tflite", "rb") as f:
    model_baytlar = f.read()

# C başlık dosyası (.h) olarak yaz
with open("../firmware/model_data.h", "w") as f:
    f.write("// Otomatik üretildi: hareket tanima int8 modeli\n")
    f.write("// Kaynak: hareket_modeli_int8.tflite\n\n")
    f.write("#ifndef MODEL_DATA_H\n#define MODEL_DATA_H\n\n")
    f.write(f"const unsigned int model_data_len = {len(model_baytlar)};\n")
    f.write("const unsigned char model_data[] = {\n")

    # baytları 12'lik satırlar halinde yaz
    for i, bayt in enumerate(model_baytlar):
        if i % 12 == 0:
            f.write("  ")
        f.write(f"0x{bayt:02x}, ")
        if i % 12 == 11:
            f.write("\n")

    f.write("\n};\n\n#endif  // MODEL_DATA_H\n")

print(f"firmware/model_data.h yazıldı.")
print(f"Model {len(model_baytlar)} bayt = {len(model_baytlar)/1024:.1f} KB olarak C dizisine gömüldü.")

firmware/model_data.h yazıldı.
Model 12448 bayt = 12.2 KB olarak C dizisine gömüldü.


In [8]:
# --- ÇİP İÇİN TEST ÖRNEKLERİ ÜRET ---
# Her hareket sınıfından birer gerçek test örneği seçip C dizisine çeviriyoruz.
# Bunları çipe gömüp modele verecek, tahminini göreceğiz.

# Her sınıftan bir örnek bul (0=WALKING, 1=W_UPSTAIRS, ... 5=LAYING)
secilen_idx = []
for sinif in range(6):
    idx = np.where(y_test == sinif)[0][0]   # o sınıfın ilk test örneği
    secilen_idx.append(idx)

# Bu örnekleri int8'e kuantize et (çipteki modelin beklediği format)
giris_olcek, giris_sifir = girdi["quantization"]

with open("../wokwi_hareket/src/test_ornekleri.h", "w") as f:
    f.write("// Otomatik üretildi: her hareketten bir test örneği (int8)\n")
    f.write("#ifndef TEST_ORNEKLERI_H\n#define TEST_ORNEKLERI_H\n\n")
    f.write("#define ORNEK_SAYISI 6\n")
    f.write("#define ORNEK_UZUNLUK 768   // 128 zaman adimi x 6 kanal\n\n")

    # Gerçek etiketler (kontrol için)
    f.write("const int gercek_etiketler[ORNEK_SAYISI] = {")
    f.write(", ".join(str(int(y_test[i])) for i in secilen_idx))
    f.write("};\n\n")

    # Test verileri (int8 diziler)
    f.write("const signed char test_verileri[ORNEK_SAYISI][ORNEK_UZUNLUK] = {\n")
    for idx in secilen_idx:
        ornek = X_test[idx]                       # (128, 6)
        duz = ornek.flatten()                     # 768 sayı
        q = np.round(duz / giris_olcek + giris_sifir).astype(np.int8)
        f.write("  {")
        f.write(", ".join(str(int(v)) for v in q))
        f.write("},\n")
    f.write("};\n\n#endif\n")

print("test_ornekleri.h olusturuldu (wokwi_hareket/src/ icine).")
print("Secilen ornekler ve gercek etiketleri:")
hareket_isimleri = ["WALKING", "W_UPSTAIRS", "W_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]
for idx in secilen_idx:
    print(f"  index {idx} -> {hareket_isimleri[int(y_test[idx])]}")

test_ornekleri.h olusturuldu (wokwi_hareket/src/ icine).
Secilen ornekler ve gercek etiketleri:
  index 79 -> WALKING
  index 133 -> W_UPSTAIRS
  index 109 -> W_DOWNSTAIRS
  index 31 -> SITTING
  index 0 -> STANDING
  index 55 -> LAYING
